# BIO-NN Experiment 3: Criticality & Emergence Analysis

Test if a spiking neural network operates near a critical phase transition.
Biological brains are believed to operate at the "edge of chaos" - a critical point
between order and disorder that maximizes computational capacity.

### 3.1 Collect Spike Data from Adaptive LIF Network

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from bio_nn.neurons import create_neuron

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Create a larger network to observe emergent dynamics
neuron = create_neuron("adaptive_lif", 512).to(device)
state = neuron._get_initial_state(16, device)

print("Collecting spike data (200 time steps)...")
spike_log = []
mem_log = []

for i in range(200):
    # Gradually increase input to drive the network from silent to active
    x = torch.randn(16, 512).to(device) * (0.3 + 0.7 * (i / 200))
    spikes, mem, state = neuron(x, state)
    spike_log.append(spikes.cpu())
    mem_log.append(mem.cpu())

spike_tensor = torch.stack(spike_log)
mem_tensor = torch.stack(mem_log)

print(f"Spike tensor: {spike_tensor.shape} (time, batch, neurons)")
print(f"Overall spike rate: {spike_tensor.float().mean().item():.4f}")
print(f"Max spike rate: {max(s.float().mean().item() for s in spike_log):.4f}")

### 3.2 Spike Raster & Membrane Dynamics

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Spike raster
axes[0, 0].imshow(spike_tensor[:, 0, :200].numpy().T, aspect='auto', cmap='hot', interpolation='nearest')
axes[0, 0].set_title("Spike Raster (first 200 neurons)", fontweight='bold')
axes[0, 0].set_xlabel("Time Step")
axes[0, 0].set_ylabel("Neuron Index")

# Membrane potential for 4 neurons
for n in range(4):
    axes[0, 1].plot(mem_tensor[:, 0, n].numpy(), linewidth=0.8, label=f'neuron {n}')
axes[0, 1].set_title("Membrane Potential (4 neurons)", fontweight='bold')
axes[0, 1].set_xlabel("Time Step")
axes[0, 1].set_ylabel("Voltage")
axes[0, 1].legend(fontsize=9)
axes[0, 1].grid(True, alpha=0.3)

# Spike rate over time
rates = [s.float().mean().item() for s in spike_log]
axes[1, 0].plot(rates, 'g-', linewidth=1.5)
axes[1, 0].axhline(y=np.mean(rates), color='r', linestyle='--', alpha=0.7, label=f'mean={np.mean(rates):.4f}')
axes[1, 0].set_title("Firing Rate Over Time", fontweight='bold')
axes[1, 0].set_xlabel("Time Step")
axes[1, 0].set_ylabel("Spike Rate")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Spike rate distribution
axes[1, 1].hist(rates, bins=30, color='steelblue', edgecolor='black', alpha=0.7)
axes[1, 1].axvline(x=np.mean(rates), color='red', linestyle='--', label=f'mean={np.mean(rates):.4f}')
axes[1, 1].set_title("Spike Rate Distribution", fontweight='bold')
axes[1, 1].set_xlabel("Spike Rate")
axes[1, 1].set_ylabel("Count")
axes[1, 1].legend()

plt.suptitle("Network Dynamics Overview", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('dynamics_overview.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.3 Branching Ratio Analysis

The **branching ratio** (sigma) measures how activity propagates through the network:
- sigma < 1: Subcritical (activity dies out, ordered)
- sigma ~ 1: Critical (edge of chaos, maximum computation)
- sigma > 1: Supercritical (activity explodes, chaotic)

In [ ]:
from bio_nn.emergence.criticality import compute_branching_ratio

# compute_branching_ratio expects (time, neurons) - use first batch
branching_mean, branching_std = compute_branching_ratio(spike_tensor[:, 0, :])

print(f"Branching Ratio: {branching_mean:.4f} +/- {branching_std:.4f}")
print(f"Target: ~1.0 for criticality")

if abs(branching_mean - 1.0) < 0.1:
    print("  -> Network is near CRITICAL point!")
elif branching_mean < 1.0:
    print("  -> Network is SUBCRITICAL (ordered)")
else:
    print("  -> Network is SUPERCRITICAL (chaotic)")

### 3.4 Avalanche Size Distribution

At criticality, neural avalanches follow a **power-law distribution**:
P(s) ~ s^(-tau) where tau ~ 1.5 (experimental finding in biological brains).

In [ ]:
from bio_nn.emergence.criticality import _detect_avalanches

# _detect_avalanches expects (time, neurons) - use first batch
sizes, durations = _detect_avalanches(spike_tensor[:, 0, :].numpy())

if len(sizes) > 10:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Size distribution
    axes[0].hist(sizes, bins=50, color='steelblue', edgecolor='black', alpha=0.7, density=True)
    axes[0].set_xscale('log')
    axes[0].set_yscale('log')
    axes[0].set_title("Avalanche Size Distribution (log-log)", fontweight='bold')
    axes[0].set_xlabel("Avalanche Size")
    axes[0].set_ylabel("Probability Density")
    axes[0].grid(True, alpha=0.3)

    # Add power-law reference line
    x_ref = np.logspace(np.log10(max(sizes.min(), 1)), np.log10(sizes.max()), 50)
    y_ref = x_ref ** (-1.5) * 0.1
    axes[0].plot(x_ref, y_ref, 'r--', linewidth=2, label='Power law (tau=1.5)')
    axes[0].legend()

    # Duration distribution
    axes[1].hist(durations, bins=50, color='coral', edgecolor='black', alpha=0.7, density=True)
    axes[1].set_xscale('log')
    axes[1].set_yscale('log')
    axes[1].set_title("Avalanche Duration Distribution (log-log)", fontweight='bold')
    axes[1].set_xlabel("Avalanche Duration")
    axes[1].set_ylabel("Probability Density")
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('avalanche_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f"Number of avalanches: {len(sizes)}")
    print(f"Mean size: {np.mean(sizes):.2f}, std: {np.std(sizes):.2f}")
    print(f"Mean duration: {np.mean(durations):.2f}, std: {np.std(durations):.2f}")
else:
    print(f"Not enough avalanches detected ({len(sizes)}). Try adjusting input intensity.")

### 3.5 Lyapunov Exponent (Chaos Detection)

The **Lyapunov exponent** measures sensitivity to initial conditions:
- lambda > 0: Chaotic (unpredictable long-term)
- lambda < 0: Stable (converges to fixed point)
- lambda ~ 0: Edge of chaos (critical)

In [ ]:
from bio_nn.emergence.complexity import estimate_lyapunov_exponents

# estimate_lyapunov_exponents expects (time, neurons)
lyapunov_max, lyapunov_all, is_chaotic = estimate_lyapunov_exponents(
    spike_tensor[:, 0, :], dt=1.0
)

print(f"Largest Lyapunov Exponent: {lyapunov_max:.6f}")
print(f"All exponents: {lyapunov_all}")
print(f"Is chaotic: {is_chaotic}")

if lyapunov_max > 0.01:
    print("  -> Network is CHAOTIC")
elif lyapunov_max < -0.01:
    print("  -> Network is STABLE")
else:
    print("  -> Network is at EDGE OF CHAOS (critical)")

### 3.6 Summary Dashboard

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# 1. Branching ratio gauge
ax = axes[0, 0]
ax.barh([0], [branching_mean], color='green' if abs(branching_mean - 1.0) < 0.1 else 'orange', height=0.3)
ax.axvline(x=1.0, color='red', linestyle='--', linewidth=2, label='Critical (sigma=1)')
ax.set_xlim([0, 2])
ax.set_title(f"Branching Ratio: {branching_mean:.4f}", fontweight='bold')
ax.set_xlabel("Sigma")
ax.legend()

# 2. Lyapunov gauge
ax = axes[0, 1]
color = 'green' if abs(lyapunov_max) < 0.01 else ('red' if lyapunov_max > 0 else 'blue')
ax.barh([0], [abs(lyapunov_max)], color=color, height=0.3)
ax.axvline(x=0.01, color='red', linestyle='--', linewidth=2, label='Critical boundary')
ax.set_title(f"Lyapunov Exponent: {lyapunov_max:.6f}", fontweight='bold')
ax.set_xlabel("|Lambda|")
ax.legend()

# 3. Firing rate time course
ax = axes[0, 2]
ax.plot(rates, 'g-', linewidth=1)
ax.axhline(y=np.mean(rates), color='r', linestyle='--', alpha=0.7)
ax.fill_between(range(len(rates)), rates, alpha=0.3)
ax.set_title("Firing Rate Over Time", fontweight='bold')
ax.set_xlabel("Time Step")
ax.set_ylabel("Spike Rate")

# 4. Spike raster
ax = axes[1, 0]
ax.imshow(spike_tensor[:, 0, :128].numpy().T, aspect='auto', cmap='hot')
ax.set_title("Spike Raster", fontweight='bold')
ax.set_xlabel("Time")
ax.set_ylabel("Neuron")

# 5. Membrane variance over time
ax = axes[1, 1]
mem_vars = [m.var().item() for m in mem_log]
ax.plot(mem_vars, 'b-', linewidth=1)
ax.set_title("Membrane Variance Over Time", fontweight='bold')
ax.set_xlabel("Time Step")
ax.set_ylabel("Variance")
ax.grid(True, alpha=0.3)

# 6. Summary text
ax = axes[1, 2]
ax.axis('off')
status_b = 'CRITICAL' if abs(branching_mean - 1.0) < 0.1 else 'NON-CRITICAL'
status_l = 'EDGE OF CHAOS' if abs(lyapunov_max) < 0.01 else ('CHAOTIC' if lyapunov_max > 0 else 'STABLE')
summary_text = (
    f"CRITICALITY ANALYSIS SUMMARY\n"
    f"{'=' * 35}\n\n"
    f"Branching Ratio:  {branching_mean:.4f}\n"
    f"  Status: {status_b}\n\n"
    f"Lyapunov:         {lyapunov_max:.6f}\n"
    f"  Status: {status_l}\n\n"
    f"Mean Spike Rate:  {np.mean(rates):.4f}\n"
    f"Mean Membrane:    {mem_tensor.mean().item():.4f}\n"
    f"Membrane Std:     {mem_tensor.std().item():.4f}"
)
ax.text(0.1, 0.9, summary_text, transform=ax.transAxes, fontsize=11,
        verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.5))

plt.suptitle("Criticality & Emergence Analysis", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('criticality_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.7 Download Results

In [ ]:
from google.colab import files
for f in ['dynamics_overview.png', 'avalanche_analysis.png', 'criticality_dashboard.png']:
    try:
        files.download(f)
    except:
        print(f"{f} not found")